# Molecular Design VAE — Moderate Mode

**~25k ZINC molecules · 250 epochs · CPU · ~60-90 min**

Run cells in order. Downloads ZINC data on first run (~2 min).

## Cell 1 — Setup

In [ ]:
import os, subprocess

# Clone repo
if not os.path.exists('molecular-design-vae'):
    subprocess.run(['git', 'clone', 'https://github.com/Kaur-Simarpreet/molecular-design-vae.git'], check=True)

# Change directory — os.chdir works for all subsequent commands
os.chdir('molecular-design-vae')
print('Directory:', os.getcwd())

# Install dependencies
subprocess.run(['pip', 'install', '-q', 'torch', 'selfies',
                'flask', 'flask-cors', 'scipy', 'requests'])
subprocess.run(['pip', 'install', '-q', 'rdkit'])
print('Setup complete')


## Cell 2 — Train (~60-90 min)

First run downloads ZINC-250K (~2 min, cached after that).

In [ ]:
import os
if not os.path.exists('train_vae_extended.py'):
    os.chdir('molecular-design-vae')
print('Training from:', os.getcwd())
!python train_vae_extended.py --mode moderate


## Cell 3 — Start server + get live URL

No signup, no account, no password.

In [ ]:
import os, subprocess, threading, time, re, urllib.request

# Confirm directory
if not os.path.exists('serve.py'):
    os.chdir('molecular-design-vae')
REPO_DIR = os.getcwd()
print('Running from:', REPO_DIR)

# Verify model files exist
required = ['vae.pt', 'tokenizer.pkl', 'latents.pt', 'config.json']
missing = [f for f in required if not os.path.exists(f'saved_model/{f}')]
if missing:
    raise FileNotFoundError(f'Missing: {missing} — re-run training cell first')
print('Model files OK')

# Start server
subprocess.Popen(
    ['python', 'serve.py', '--host', '0.0.0.0', '--port', '5000'],
    cwd=REPO_DIR
)
print('Server starting — waiting 20 seconds...')
time.sleep(20)

# Verify server is up
for attempt in range(5):
    try:
        r = urllib.request.urlopen('http://localhost:5000/health', timeout=5)
        print('Server UP:', r.read().decode()[:80])
        break
    except Exception as e:
        print(f'Attempt {attempt+1}/5: {e}')
        if attempt < 4: time.sleep(5)
        else: raise RuntimeError('Server failed — check output above')

# Download cloudflared
subprocess.run(['wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/tmp/cloudflared'], check=True)
subprocess.run(['chmod', '+x', '/tmp/cloudflared'])

# Start tunnel
proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
)
print('Starting tunnel...')
for line in proc.stderr:
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m:
        print('\n' + '='*50)
        print('OPEN THIS URL IN YOUR BROWSER:')
        print(m.group(0))
        print('='*50)
        print('All 8 tabs work. No password needed.')
        break


## Cell 4 — Download trained model (optional)

In [ ]:
import os, shutil
from google.colab import files
if not os.path.exists('saved_model'):
    os.chdir('molecular-design-vae')
shutil.make_archive('/tmp/saved_model', 'zip', 'saved_model')
print('Archive size:', os.path.getsize('/tmp/saved_model.zip'), 'bytes')
files.download('/tmp/saved_model.zip')
